## Perform trajectory analysis on VASA data

In [ ]:
import os
import sys
import random
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", UserWarning)

random.seed(10)

from datetime import datetime
from pathlib import Path
from anndata import AnnData
import numpy as np
import pandas as pd
import scanpy as sc
import scFates as scf
#import palantir
import logging
import cellrank as cr
import scvelo as scv
import matplotlib.pyplot as plt
import seaborn
seaborn.reset_orig()
%matplotlib inline

logging.getLogger('matplotlib.font_manager').disabled = True

sc.set_figure_params(figsize=(10, 10))
scf.set_figure_pubready()

figdir = "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/VASA"
sc.settings.figdir = figdir

## Load data

In [ ]:
# Load the dataset
adata_orig = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.h5ad")

# Remove SMOC2+ cells
adata_orig = adata_orig[adata_orig.obs["cell_type_initial"] != "SMOC2+ cells"]

In [ ]:
# Make copy for downstream
adata = adata_orig.copy()

In [ ]:
# Remove genes expressed in fewer than 3 cells
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
# Standard preprocessing
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

In [ ]:
adata

In [ ]:
sc.pp.neighbors(adata, n_pcs = 30, n_neighbors = 20)
sc.tl.umap(adata)

## Run Diffusion Map & Set Root Node

In [ ]:
sc.tl.diffmap(adata)

## Plot Diffusion Map

In [ ]:
elem = adata.obs["DT"].sort_values().index[9]
root_ixs = list(adata.obs.index).index(elem)

In [ ]:
scv.pl.scatter(
    adata,
    basis="diffmap",
    c=["DT", root_ixs],
    legend_loc="right",
    components=["2, 3"],
    show=False,
    projection="2d"
)

adata.uns["iroot"] = root_ixs

## Compute pseudotime

In [ ]:
sc.tl.dpt(adata)

## Plot DT & pseudotime

In [ ]:
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

def r2(x, y):
    return stats.pearsonr(x, y)[0] ** 2

# Assuming adata is already defined and contains the necessary data
df = adata.obs[["DT", "dpt_pseudotime"]]
df = df[df["DT"].abs() < 5]

# Create the joint plot with regression
g = sns.jointplot(x=df["DT"], y=df["dpt_pseudotime"], kind="reg", palette='tab10')

# Annotate with R^2 value
r_squared = r2(df["DT"], df["dpt_pseudotime"])
plt.annotate(f'$R^2 = {r_squared:.2f}$', xy=(0.1, 0.9), xycoords='axes fraction')

plt.show()

In [ ]:
sc.pl.embedding(
    adata,
    basis="X_umap",
    color=["DT", "dpt_pseudotime"]
)

## Run CellRank

In [ ]:
from cellrank.kernels import PseudotimeKernel

pk = PseudotimeKernel(adata, time_key="dpt_pseudotime")
pk.compute_transition_matrix()


In [ ]:
fig, axs = plt.subplots(1,1,figsize=(10,10))

pk.plot_projection(basis="X_umap", color="cell_type_initial", recompute=True, ax=axs, show=False)

In [ ]:
g = cr.estimators.GPCCA(pk)
print(g)

In [ ]:
g.fit(cluster_key="cell_type_initial", n_states=[6, 8])
g.plot_macrostates(which="all", discrete=True, legend_loc="right", s=100)

In [ ]:
g.predict_terminal_states()
g.set_terminal_states(["Late ECs", "Late X cells", "D cells", "K cells"])
g.plot_macrostates(which="terminal", legend_loc="right", s=100)

In [ ]:
g.plot_coarse_T()

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(same_plot=False)

In [ ]:
fig, axs = plt.subplots(1,1,figsize=(10,10))

g.plot_fate_probabilities(same_plot=True, show=False, ax=axs)


In [ ]:
g.compute_absorption_times()

## Plot fate probabilities vs DT

In [ ]:
# Assuming adata is already defined and contains the necessary data
df = adata.obs[["DT", "term_states_fwd_probs", "cell_type_initial"]]

In [ ]:
# Calculate the mean DT for each cell type
mean_dt = df.groupby('cell_type_initial')['DT'].mean().sort_values()

# Reorder the categories of cell_type_initial based on the mean DT
df['cell_type_initial'] = pd.Categorical(df['cell_type_initial'], categories=mean_dt.index.tolist(), ordered=True)

# Plotting
plt.figure(figsize=(12, 6))
sns.boxplot(x='cell_type_initial', y='DT', data=df)
plt.xticks(rotation=45)
plt.xlabel('Cell Type')
plt.ylabel('Differentiation Time (DT)')
plt.title('Differentiation Time by Cell Type')
plt.tight_layout()
plt.show()

In [ ]:
# Plotting
plt.figure(figsize=(12, 6))
sns.boxplot(x='cell_type_initial', y='term_states_fwd_probs', data=df)
plt.xticks(rotation=45)
plt.xlabel('Cell Type')
plt.ylabel('Terminal fate probability')
plt.title('Terminal fate probabilities by Cell Type')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming df is your DataFrame
ds = df.groupby("cell_type_initial")[["DT", "term_states_fwd_probs"]].agg(
    mean_DT=('DT', 'mean'),
    mean_term_states_fwd_probs=('term_states_fwd_probs', 'mean'),
    sem_DT=('DT', 'sem'),
    sem_term_states_fwd_probs=('term_states_fwd_probs', 'sem')
).reset_index()

# Plotting
plt.figure(figsize=(12, 6))
scatter = sns.scatterplot(x='mean_DT', y='mean_term_states_fwd_probs', hue="cell_type_initial", data=ds)

# Get the color palette
palette = sns.color_palette(n_colors=ds['cell_type_initial'].nunique())
colors = dict(zip(ds['cell_type_initial'].unique(), palette))

# Adding error bars with corresponding colors
for i, row in ds.iterrows():
    plt.errorbar(x=row['mean_DT'], y=row['mean_term_states_fwd_probs'],
                 xerr=row['sem_DT'], yerr=row['sem_term_states_fwd_probs'],
                 fmt='o', ecolor=colors[row['cell_type_initial']], elinewidth=2, capsize=4, color=colors[row['cell_type_initial']])

plt.xlabel('Mean DT')
plt.ylabel('Mean Term States Fwd Probs')
plt.title('Mean Values with Standard Error Bars')
plt.tight_layout()
plt.show()


## Compute lineage drivers

In [ ]:
# Compute lineage drivers
delta_df = g.compute_lineage_drivers(cluster_key="cell_type_initial")

In [ ]:
# Export 
delta_df.to_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.cellrank.lineage.drivers.csv")

In [ ]:
delta_df.columns

In [ ]:
genes_oi = {
    "X cells": list(delta_df.sort_values(by="Late X cells_ci_high", ascending=False).head(10).index),
    "ECs": list(delta_df.sort_values(by="Late ECs_ci_high", ascending=False).head(10).index)
}

In [ ]:
# compute mean gene expression across all cells
adata.var["mean expression"] = adata.X.A.mean(axis=0)

In [ ]:
# visualize in a scatter plot
g.plot_lineage_drivers_correlation(
    lineage_x="Late X cells",
    lineage_y="Late ECs",
    adjust_text=True,
    gene_sets=genes_oi,
    color="mean expression",
    legend_loc="none",
    figsize=(5, 5),
    dpi=150,
    fontsize=9,
    size=50,
    show=False
)

In [ ]:
adata.write("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.cellrank.h5ad")

## Load back into memory

In [ ]:
# Load data
adata = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.cellrank.h5ad")
adata = adata.raw.to_adata()

# Add approriate colors for the terminal states
adata.uns['term_states_fwd_colors'] = ["#ee8865","#85a7cf", "#6ac077", "#58B6D7"]

# Standard preprocessing
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

In [ ]:
sc.pl.umap(adata, color=["term_states_fwd"], wspace=0.4)

In [ ]:
# Define model using adata with normalized and log-transformed data
model = cr.models.GAMR(adata, n_knots=6, smoothing_penalty=10.0)

In [ ]:
# Define the perturbed TFs
perturbed_tfs = 'Nr2e3,Prdm16,Insm1,Lmx1b,Percc1,Tead1,Klf4,Pou2af3,Npas2,Atoh1,Rfx3,Rora,Setbp1,Thrb,Zbtb18,Zbtb46,Zbtb7c,Zkscan1,Znf326,Znf445,Znf608,Znf704,Znf711,Bhlhe40,Mxd1,Spib,Hmx2,Tcf7,Neurog3,Neurod1,Neurod2,Gfi1,Foxa1,Foxa2,Arx,Mnx1,Isl1,Pax4,Pax6,Nkx6-1,Nkx2-2,Pdx1,Hes1,Hhex,Sox4,Znf800,Lmx1a,Runx1t1,Tox3,Zcchc12,Bambi,Rfx6,Arnt2,Egr4,Prox1,Id1,Id3,Hets1,Hetv4,Hdac9,Hhmgb3,Spdef,Dach2,Hes2,Hes4,Myt1l,Pou2af2,Arid5b,Csrnp3,Etv1,Fos,Foxn3,Gata4,Glis3,Hnf4g,Gtf2ird1,Cxxc4,Hif1a,Jazf1,Klf12,L3mbtl3,Lcorl,Meis3'
perturbed_tfs = perturbed_tfs.split(',')
perturbed_tfs = [c.upper() for c in perturbed_tfs]

# Check which perturbed TFs are missing from the dataset
print(f"TFs missing from the dataset: {[c for c in perturbed_tfs if c not in adata.var_names]}")

# Filter the perturbed TFs to only include those present in the dataset
perturbed_tfs = [c for c in perturbed_tfs if c in adata.var_names]
print(f"Number of perturbed TFs found in the dataset: {len(perturbed_tfs)}")

In [ ]:
# Define custom gene set to plot
genes = [
    "ARX", "PERCC1", "LMX1A", "HHEX", "PDX1", "PLAGL1",
    "TEAD1", "PRDM16", "THRB", "CXXC4", "RFX3", "LMX1B",
    "GHRL", "TPH1", "SST", "CCK", "GIP", "NTS", "SCT", "GCG", 
    "PYY", "MLN", "YAP1", "HES6", "HES4", "ONECUT3", "ASCL2",
    "FEV"
]
genes = [g for g in genes if g in adata.var_names]

# Add perturbed TFs to the gene set
genes = list(set(genes) | set(perturbed_tfs))

# Print number of genes in the final gene set
print(f"Number of genes in the final gene set: {len(genes)}")

In [ ]:
# Define the output directory
outfile_dir = "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/VASA/cellrank_trends_v3/"
os.makedirs(outfile_dir, exist_ok=True)

# Shared timestamp for all files from this run
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")

for gene in genes:
    fig = cr.pl.gene_trends(
        adata,
        model=model,
        genes=[gene],              # one gene at a time
        same_plot=True,
        ncols=1,
        time_key="dpt_pseudotime",
        hide_cells=True,
        legend_loc=None,
        weight_threshold=(1e-3, 1e-3),
        return_figure=True
    )

    # Adjust axes
    for ax in fig.axes:
        ax.set_xlabel("Pseudotime")
        ax.set_ylabel("Expression")
        ax.set_ylim(bottom=0)
        ax.grid(False)

        # optional: remove top/right spines
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.tight_layout()

    outfile_path = os.path.join(
        outfile_dir,
        f"CellRank_trend_{gene}_{current_datetime}.pdf"
    )

    fig.savefig(outfile_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

print(f"Saved {len(genes)} gene trend plots to: {outfile_dir}")

## Further process scFates

In [ ]:
adata = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.cellrank.h5ad")

In [ ]:
adata.obsm["to_terminal_states"] = adata.obsm["term_states_fwd_memberships"]

In [ ]:
scf.tl.cellrank_to_tree(adata,time="dpt_pseudotime",Nodes=200,seed=10)
scf.pl.graph(adata)

In [ ]:
scf.tl.cleanup(adata)

In [ ]:
scf.tl.root(adata,170)

In [ ]:
scf.tl.pseudotime(adata,n_jobs=20,seed=42)

In [ ]:
fig, axs = plt.subplots(1,1,figsize=(10,10))

scf.pl.trajectory(adata,color_cells="t", ax=axs, show=False)

In [ ]:
fig, axs = plt.subplots(1,1,figsize=(10,10))

scf.pl.trajectory(adata,color_cells="DT", ax=axs, show=False)

In [ ]:
scf.tl.dendrogram(adata)

In [ ]:
scf.pl.dendrogram(adata,color="cell_type_initial", show=False)

In [ ]:
scf.pl.dendrogram(adata,color_milestones=True,color="milestones")

In [ ]:
adata.write("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.h5ad")

# Plot fates 

In [ ]:
adata = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.h5ad")
adata  = adata.raw.to_adata()

In [ ]:
# Remove genes expressed in fewer than 3 cells
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
# Standard preprocessing
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

In [ ]:
# Add umap embedding from adata_orig back to adata
adata.obsm["X_umap"] = adata_orig[adata.obs_names].obsm["X_umap"]

In [ ]:
# Load mapping
mapping = dict(zip(["100", "140", "186", "120", "59", "170", "165", "17"], 
                   ["X cells","D cells","X/D/I/K cells","EC/X/D/I/K cells",
                    "X/D/I/K cells","NGN3+ cells", "EC cells", "I/K cells"]))

In [ ]:
# Load mapping
#mapping = dict(zip(["132", "137", "187", "219", "228", "232", "244", "269"], 
#                   ["X cells","D cells","D/I/K cells","EC/X/D/I/K cells",
#                    "X/D/I/K cells","NGN3+ cells", "EC cells", "I/K cells"]))

# Transfer annotation
adata.obs["milestones_anno"] = adata.obs["milestones"].map(mapping)

In [ ]:
ct_palette = {
    # Pure populations
    "NGN3+ cells": "#9467bd",   # purple (progenitors)
    "EC/X/D/I/K cells": "#fdae61",    # warm orange blend
    "EC cells": "#d62728",      # strong red
    "X/D/I/K cells": "#c2a5cf",       # lavender blend
    "X cells": "#e377c2",       # magenta
    "D cells": "#17becf",       # cyan
    "I/K cells": "#bcbd22",     # olive
}

In [ ]:
adata.obs["milestones_anno"] = adata.obs["milestones_anno"].astype("category")
adata.obs["milestones_anno"] = adata.obs["milestones_anno"].cat.reorder_categories(ct_palette.keys())

In [ ]:
fig, axs = plt.subplots(1,1,figsize=(5,5))

scf.pl.trajectory(adata,color_cells="milestones_anno", palette=ct_palette, ax=axs, show=False)

fig.savefig(Path(sc.settings.figdir) / "VASA_trajectory_milestones_anno.pdf")

In [ ]:
perturbed_tfs = 'Nr2e3,Prdm16,Insm1,Lmx1b,Percc1,Tead1,Klf4,Pou2af3,Npas2,Atoh1,Rfx3,Rora,Setbp1,Thrb,Zbtb18,Zbtb46,Zbtb7c,Zkscan1,Znf326,Znf445,Znf608,Znf704,Znf711,Bhlhe40,Mxd1,Spib,Hmx2,Tcf7,Neurog3,Neurod1,Neurod2,Gfi1,Foxa1,Foxa2,Arx,Mnx1,Isl1,Pax4,Pax6,Nkx6-1,Nkx2-2,Pdx1,Hes1,Hhex,Sox4,Znf800,Lmx1a,Runx1t1,Tox3,Zcchc12,Bambi,Rfx6,Arnt2,Egr4,Prox1,Id1,Id3,Hets1,Hetv4,Hdac9,Hhmgb3,Spdef,Dach2,Hes2,Hes4,Myt1l,Pou2af2,Arid5b,Csrnp3,Etv1,Fos,Foxn3,Gata4,Glis3,Hnf4g,Gtf2ird1,Cxxc4,Hif1a,Jazf1,Klf12,L3mbtl3,Lcorl,Meis3'
perturbed_tfs = perturbed_tfs.split(',')
perturbed_tfs = [c.upper() for c in perturbed_tfs]
perturbed_tfs = [c for c in perturbed_tfs if c in adata.var_names]


In [ ]:
scf.tl.fit(adata, features=perturbed_tfs, n_jobs=20)

In [ ]:
adata = adata[:, perturbed_tfs].copy()

In [ ]:
adata.write("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.tf.perturb.fit.h5ad")

## Map fitted gene expression onto closest branchpoint

In [ ]:
adata = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.tf.perturb.fit.h5ad")

In [ ]:
# Save TFs in adata
with open("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.perturbed_tfs.txt", "w") as f:
    for tf in adata.var_names:
        f.write(f"{tf}\n")

In [ ]:
sc.pl.umap(adata, color=["milestones", "seg"], ncols=4, size=50)

In [ ]:
import random
import numpy as np
import pandas as pd
import igraph
import networkx as nx
import matplotlib.pyplot as plt

# Tree layout helper
def hierarchy_pos(G, root=None, width=1.0, vert_gap=0.2, vert_loc=0, xcenter=0.5):
    if not nx.is_tree(G):
        raise TypeError("hierarchy_pos can only be used on a tree.")

    # Pick root if not given
    if root is None:
        root = next(iter(nx.topological_sort(G))) if isinstance(G, nx.DiGraph) else random.choice(list(G.nodes))

    def recurse(node, width, vert_loc, xcenter, pos, parent=None):
        pos[node] = (xcenter, vert_loc)
        children = list(G.neighbors(node))

        # When graph is undirected, exclude parent in recursion
        if parent is not None and not isinstance(G, nx.DiGraph):
            children = [c for c in children if c != parent]

        if children:
            dx = width / len(children)
            x = xcenter - width / 2 - dx / 2
            for child in children:
                x += dx
                recurse(child, width=dx, vert_loc=vert_loc - vert_gap, xcenter=x, pos=pos, parent=node)

        return pos

    return recurse(root, width, vert_loc, xcenter, pos={})


def build_cell_lineage_graph(graph):
    milestone_dict = graph["milestones"]

    # Convert to strings for compatibility
    milestone_ids = [str(v) for v in milestone_dict.values()]
    milestone_labels = list(milestone_dict.keys())

    # Build igraph
    g = igraph.Graph(directed=True)
    g.add_vertices(len(milestone_ids))

    # Assign vertex names and labels
    g.vs["name"] = milestone_ids
    g.vs["label"] = milestone_labels

    # Build edges
    edge_tuples = graph["pp_seg"][["from", "to"]].astype(str).apply(tuple, axis=1).tolist()
    g.add_edges(edge_tuples)

    # Convert to networkx
    G = g.to_networkx()

    # Build dict to rename nodes based on milestone labels
    id_to_label = {i: milestone_labels[i] for i in range(len(milestone_ids))}

    # Relabel nodes in NetworkX
    G = nx.relabel_nodes(G, id_to_label)

    return G

In [ ]:
G = build_cell_lineage_graph(adata.uns["graph"])
pos = hierarchy_pos(G)

In [ ]:
G = nx.relabel_nodes(G, mapping)
pos = {mapping[k]: v for k, v in pos.items()}

In [ ]:
milestone_order

In [ ]:
# Extract expression as dense DataFrame + add obs grouping column
df = sc.get.obs_df(adata, list(adata.var_names))
df["milestones_anno"] = adata.obs["milestones_anno"].values

# Aggregate mean expression per seg
df_mean = df.groupby("milestones_anno").mean()

df_mean = df_mean.reindex(adata.obs["milestones_anno"].cat.categories)

In [ ]:
df_mean

In [ ]:
from collections import defaultdict

# Assign TF to node with highest expression
tf_to_node = {}
for tf in perturbed_tfs:
    if tf not in df_mean.columns:
        continue
    max_node = df_mean[tf].idxmax()
    tf_to_node[tf] = max_node

# Invert mapping to get node to TFs
node_to_tfs = defaultdict(list)
for tf, node in tf_to_node.items():
    node_to_tfs[node].append(tf)

In [ ]:
figsize = (10, 8)
fontsize = 12

# Compute TF counts per node
tf_counts = {node: len(node_to_tfs.get(node, [])) for node in G.nodes()}  # ensure length, not direct number
node_sizes = [200 + 200 * tf_counts[n] for n in G.nodes()]

# Compute label texts
labels = {node: f"{node}\n({tf_counts[node]} TFs)" for node in G.nodes()}

plt.figure(figsize=figsize)

# Draw graph
nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color="grey",
    edgecolors="black"
)
nx.draw_networkx_labels(G, pos, font_size=fontsize)

nx.draw_networkx_edges(
    G,
    pos,
    arrowstyle="-|>",
    arrows=True,
    arrowsize=12,
    connectionstyle="arc3,rad=-0.12",
    width=1.3
)

# --- Add TF text boxes ---
for node, (x, y) in pos.items():
    tf_list = node_to_tfs.get(node, [])
    
    if len(tf_list) == 0:
        continue
    
    # multiline TF string
    text = "\n".join(tf_list)
    
    plt.text(
        x + 0.03, y, text,  # x-offset to avoid overlap; adjust if needed
        fontsize=fontsize,
        va='center',
        ha='left',
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="black", boxstyle="round,pad=0.3")
    )

plt.axis("off")
plt.tight_layout()
#plt.savefig(f"{figdir}/VASA_perturbed_TF_distribution_tree.pdf", format='pdf', bbox_inches='tight')

## Plot heatmap of mean expression per milestone for perturbed TFs


In [ ]:
def order_rows_by_max(expr: pd.DataFrame) -> list:
    max_group = expr.idxmax(axis=1)  # gene → column where max occurs
    ordered = []
    for group in expr.columns:
        # genes peaking in this group, sorted by descending expression
        genes = expr.loc[max_group == group, group].sort_values(ascending=False).index
        ordered.extend(genes)
    return ordered

row_order = order_rows_by_max(df_mean.T)
df_norm = df_mean[row_order].T.copy()
df_norm = df_norm.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=1).T

In [ ]:
# Save df_norm as CSV
df_norm.to_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.perturbed_tfs.expression.heatmap.csv")

In [ ]:
import marsilea as ma
import marsilea.plotter as mp

h = ma.Heatmap(data=df_norm, width=12, height=6, cmap="RdPu")
h.add_right(mp.Labels(df_norm.index))
h.add_bottom(mp.Labels(df_norm.columns, rotation=90))
h.render()
#plt.savefig(f"{figdir}/VASA_perturbed_TF_expression_heatmap.pdf", format='pdf', bbox_inches='tight')